#  ROL 2 – Analista de Calidad - Validación y Detección de Patrones
## Caso SUNBURST – SolarWinds 
**Curso:** Gestión de Datos  
**Facultad:** Ingeniería – Universidad de San Buenaventura  
**Marco de Referencia:** DAMA DMBOK (Dimensiones de Calidad)  

---
### Contexto del Análisis
Como Rol 2, este notebook implementa el control de calidad de datos sobre el dataset sintético de ciberseguridad (`eventos_seguridad.csv`) y realiza la detección de anomalías cruzando la información con los DataFrames generados por el Rol 1 (`instalaciones.csv` y `versiones_software.csv`). El objetivo es aislar el impacto técnico real del ataque SUNBURST frente al ruido mediático.

In [ ]:
# ── 1. IMPORTACIONES Y AJUSTE DE SEMILLA ─────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from faker import Faker

# Garantizamos reproducibilidad según la guía del laboratorio
SEED = 42
np.random.seed(SEED)
fake = Faker("es_MX")
fake.seed_instance(SEED)

## 2. Ingesta e Integración de Datos (Cruce con Rol 1)
Cargamos las fuentes de datos generadas por el Diseñador de Datos (Rol 1) para evaluar las llaves foráneas y validar las versiones de software comprometidas.


In [ ]:
# Carga de archivos del Rol 1
df_instalaciones = pd.read_csv('data/instalaciones.csv')
df_versiones = pd.read_csv('data/versiones_software.csv')

# Extraemos la lista de IDs de instalación válidos para la regla de Integridad Referencial
lista_ids_validos = df_instalaciones['instalacion_id'].tolist()
print(f"✔ Instalaciones válidas cargadas: {len(lista_ids_validos)} registros.")

## 3. Generación del Dataset de Eventos de Seguridad (Tabla 4)
Simulamos 200 registros de eventos de red. Introducimos intencionalmente impurezas y fallos de estructura para evaluar el comportamiento de nuestras reglas de calidad DAMA.

In [ ]:
ids_eventos = []
ids_instalaciones = []
fechas = []
tipos = []
severidades = []

opciones_eventos = ['Tráfico de Red Anómalo', 'Acceso no Autorizado', 'Actualización Sistema', 'Ping de Rutina']
opciones_severidad = ['Baja', 'Media', 'Alta', 'Crítica']

for i in range(1, 201):
    ids_eventos.append("EVT-" + str(i).zfill(4))

    # Impureza de Integridad Referencial: 5 registros apuntan a ID inexistente
    if i <= 5:
        ids_instalaciones.append("INST-9999")
    else:
        ids_instalaciones.append(np.random.choice(lista_ids_validos))

    fechas.append(fake.date_between_dates(date_start=pd.to_datetime('2020-01-01'), date_end=pd.to_datetime('2020-12-31')))
    tipos.append(np.random.choice(opciones_eventos))
    severidades.append(np.random.choice(opciones_severidad))

df_eventos = pd.DataFrame({
    'evento_id': ids_eventos,
    'instalacion_id': ids_instalaciones,
    'timestamp': fechas,
    'tipo_evento': tipos,
    'severidad': severidades
})

# Introducción de impurezas adicionales de forma controlada
df_eventos.loc[10:13, 'tipo_evento'] = None                    # 4 nulos (Completitud)
df_eventos.loc[50, 'timestamp'] = pd.to_datetime('2035-05-17') # 1 fecha futura (Exactitud)

print(f"✔ DataFrame 'eventos_seguridad' creado con {len(df_eventos)} registros.")

## 4. Algoritmo de Detección de Anomalías (Aislamiento de SUNBURST)
Aplicamos un cruce de datos multivariable utilizando las columnas `instalacion_id` y `version_cod` provistas por el Rol 1. La regla lógica establece que una anomalía real se confirma únicamente si se presenta un evento de 'Tráfico de Red Anómalo', con severidad 'Alta' o 'Crítica', en una infraestructura cuyo software contiene la vulnerabilidad inyectada.

In [ ]:
# Unión con la topología e inventario del Rol 1
df_cruce1 = df_eventos.merge(df_instalaciones, on='instalacion_id', how='left')
df_cruce_final = df_cruce1.merge(df_versiones, on='version_cod', how='left')

# Regla lógica de detección estadística
condicion_ataque = (
    (df_cruce_final['tipo_evento'] == 'Tráfico de Red Anómalo') &
    (df_cruce_final['severidad'].isin(['Alta', 'Crítica'])) &
    (df_cruce_final['contiene_sunburst'] == True)
)

df_eventos['es_anomalo'] = condicion_ataque
df_eventos.to_csv('eventos_seguridad.csv', index=False)

print(f"✔ Detección completada. Ataques reales confirmados: {df_eventos['es_anomalo'].sum()} de 200 alertas.")

## 5. Cuadro de Mando de Calidad de Datos (DAMA DMBOK - Tabla 5)
Evaluamos cuantitativamente la sanidad del dataset frente a un umbral teórico del 100% en las cuatro dimensiones seleccionadas.

In [ ]:
total_filas = len(df_eventos)

# Métricas manuales paso a paso
cant_nulos = df_eventos['tipo_evento'].isna().sum()
porc_completitud = ((total_filas - cant_nulos) / total_filas) * 100

cant_fechas_malas = (df_eventos['timestamp'] > pd.to_datetime('2026-12-31')).sum()
porc_exactitud = ((total_filas - cant_fechas_malas) / total_filas) * 100

cant_sev_malas = sum([1 for x in df_eventos['severidad'] if x not in opciones_severidad])
porc_consistencia = ((total_filas - cant_sev_malas) / total_filas) * 100

cant_ids_malos = sum([1 for x in df_eventos['instalacion_id'] if x not in lista_ids_validos])
porc_integridad = ((total_filas - cant_ids_malos) / total_filas) * 100

df_reporte = pd.DataFrame({
    'tabla': ['eventos_seguridad'] * 4,
    'columna': ['tipo_evento', 'timestamp', 'severidad', 'instalacion_id'],
    'metrica': ['Completitud', 'Exactitud', 'Consistencia', 'Integridad Referencial'],
    'valor_actual': [f"{porc_completitud}%", f"{porc_exactitud}%", f"{porc_consistencia}%", f"{porc_integridad}%"],
    'umbral': ['100%'] * 4,
    'estado': ['Aprobado' if cant_nulos == 0 else 'Fallo',
               'Aprobado' if cant_fechas_malas == 0 else 'Fallo',
               'Aprobado' if cant_sev_malas == 0 else 'Fallo',
               'Aprobado' if cant_ids_malos == 0 else 'Fallo']
})

df_reporte.to_csv('data/reporte_calidad.csv', index=False)
df_reporte

## 6. Visualizaciones Estadísticas
Generamos las gráficas requeridas para la sustentación visual del comportamiento de los datos y el impacto del incidente.

In [ ]:
# Gráfico 1: Hipótesis de Prensa vs Evidencia Real
plt.figure(figsize=(6, 3.5))
conteos = df_eventos['es_anomalo'].value_counts()
plt.bar(['Alertas Generales', 'Ataques Reales'], [conteos.get(False, 0), conteos.get(True, 0)], color=['#2c3e50', '#e74c3c'])
plt.title('Caso SUNBURST: Ruido en Alertas vs Impacto Real')
plt.ylabel('Cantidad de Registros')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig('visualizations/grafico1_sunburst_realidad.png', bbox_inches='tight')
plt.close()

# Gráfico 4: Métricas de Calidad de Datos
plt.figure(figsize=(6, 3.5))
nombres = ['Completitud', 'Exactitud', 'Consistencia', 'Integridad']
valores = [porc_completitud, porc_exactitud, porc_consistencia, porc_integridad]
plt.bar(nombres, valores, color='#9b59b6')
plt.axhline(100, color='black', linestyle='--', alpha=0.7, label='Umbral Esperado (100%)')
plt.ylim(80, 105)
plt.title('Control de Calidad de Datos (Métricas DAMA)')
plt.ylabel('Porcentaje %')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig('visualizations/grafico4_reporte_calidad.png', bbox_inches='tight')
plt.close()

print("Gráficos generados y exportados.")